# 🩺 Health Risk Prediction using Multiple Linear Regression
**Data Scientist Portfolio Project**

---
**Objective:** Predict disease progression (health risk score) using multi-factor clinical & lifestyle variables.

**Dataset:** Diabetes Dataset (sklearn) — 442 patients, 10 baseline features, target = disease progression score

---

## Step 1: Install & Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set global plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('muted')

print('All libraries imported successfully!')

## Step 2: Load & Explore the Dataset

In [ ]:
# Load the diabetes dataset from sklearn
diabetes = load_diabetes()

# Create DataFrame with meaningful column names
feature_names = ['Age', 'Sex', 'BMI', 'BloodPressure', 'Serum_TC',
                 'LDL', 'HDL', 'TCH_Ratio', 'Glucose', 'Insulin']

df = pd.DataFrame(diabetes.data, columns=feature_names)
df['DiseaseProgression'] = diabetes.target  # Target: disease progression score

print('Dataset Shape:', df.shape)
print('\n Feature Descriptions:')
descriptions = {
    'Age': 'Age of patient (normalized)',
    'Sex': 'Sex of patient (normalized)',
    'BMI': 'Body Mass Index (normalized)',
    'BloodPressure': 'Average blood pressure (normalized)',
    'Serum_TC': 'Total serum cholesterol (normalized)',
    'LDL': 'Low-density lipoproteins (normalized)',
    'HDL': 'High-density lipoproteins (normalized)',
    'TCH_Ratio': 'Total cholesterol / HDL ratio (normalized)',
    'Glucose': 'Blood glucose level (normalized)',
    'Insulin': '6-hr serum insulin (normalized)',
    'DiseaseProgression': 'TARGET: Disease progression 1-yr after baseline'
}
for col, desc in descriptions.items():
    print(f'  {col:18s} → {desc}')

print('\nTarget Variable Range:', df['DiseaseProgression'].min(), '–', df['DiseaseProgression'].max())
df.head()

In [ ]:
# Statistical summary
print('Statistical Summary:')
df.describe().round(3)

In [ ]:
# Missing values check
print(' Missing Values:')
missing = df.isnull().sum()
if missing.sum() == 0:
    print(' No missing values found! Dataset is clean.')
else:
    print(missing[missing > 0])

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['DiseaseProgression'], bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribution of Disease Progression Score', fontweight='bold')
axes[0].set_xlabel('Disease Progression Score')
axes[0].set_ylabel('Frequency')

stats.probplot(df['DiseaseProgression'], dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: Is Target Normally Distributed?', fontweight='bold')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

skewness = df['DiseaseProgression'].skew()
print(f'Skewness of target: {skewness:.3f} (values close to 0 are ideal for linear regression)')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr_matrix = df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0,
    mask=mask,
    linewidths=0.5, linecolor='white',
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Matrix - All Features vs Disease Progression', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlating features with target
target_corr = corr_matrix['DiseaseProgression'].drop('DiseaseProgression').abs().sort_values(ascending=False)
print('\n Feature Correlations with Disease Progression (|r|):')
for feat, val in target_corr.items():
    bar = '█' * int(val * 20)
    print(f'  {feat:18s} {bar} {val:.3f}')

In [ ]:
# Scatter plots: Top 4 features vs target
top_features = target_corr.index[:4].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['DiseaseProgression'],
                    alpha=0.5, color='steelblue', edgecolor='none', s=40)
    # Regression line
    m, b = np.polyfit(df[feat], df['DiseaseProgression'], 1)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    axes[i].plot(x_line, m * x_line + b, color='crimson', linewidth=2, label='Trend')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Disease Progression')
    r_val = df[feat].corr(df['DiseaseProgression'])
    axes[i].set_title(f'{feat} vs Progression  (r = {r_val:.3f})', fontweight='bold')
    axes[i].legend()

plt.suptitle('Top 4 Features: Linear Relationship with Disease Progression', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Data Preprocessing

In [ ]:
# The sklearn diabetes dataset is already normalized (mean=0, std=1)
# But we'll demonstrate full pipeline for real-world applicability

X = df.drop('DiseaseProgression', axis=1)
y = df['DiseaseProgression']

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(' Data Split Summary:')
print(f'  Training samples : {X_train.shape[0]}')
print(f'  Test samples     : {X_test.shape[0]}')
print(f'  Features         : {X_train.shape[1]}')
print('\n Feature scaling applied (StandardScaler)')
print('   Mean after scaling:', np.round(X_train_scaled.mean(axis=0), 3))
print('   Std after scaling :', np.round(X_train_scaled.std(axis=0), 3))

## Step 5: Build Multiple Linear Regression Model

The model equation:
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_{10} x_{10}$$

Where $y$ = Disease Progression Score and $x_i$ = clinical/lifestyle features

In [ ]:
# Train Multiple Linear Regression
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print(' Model trained successfully!')
print(f'\n Model Equation:')
print(f'   Intercept (β₀): {model.intercept_:.4f}')
print('\n   Coefficients:')
for name, coef in zip(feature_names[:-1], model.coef_):
    direction = '↑' if coef > 0 else '↓'
    print(f'   {direction} {name:18s}: {coef:+.4f}')

## Step 6: Model Evaluation

In [ ]:
# Predictions
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

# Metrics
train_r2   = r2_score(y_train, y_pred_train)
test_r2    = r2_score(y_test, y_pred_test)
train_mse  = mean_squared_error(y_train, y_pred_train)
test_mse   = mean_squared_error(y_test, y_pred_test)
train_rmse = np.sqrt(train_mse)
test_rmse  = np.sqrt(test_mse)
test_mae   = mean_absolute_error(y_test, y_pred_test)

# Cross-validation
cv_scores = cross_val_score(model, scaler.fit_transform(X), y, cv=5, scoring='r2')

print('=' * 50)
print('        MODEL PERFORMANCE REPORT')
print('=' * 50)
print(f'  R² Score    (Train): {train_r2:.4f}')
print(f'  R² Score    (Test) : {test_r2:.4f}  ← {test_r2*100:.1f}% variance explained')
print(f'  MSE         (Test) : {test_mse:.2f}')
print(f'  RMSE        (Test) : {test_rmse:.2f}')
print(f'  MAE         (Test) : {test_mae:.2f}')
print(f'  Cross-Val R² (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=' * 50)

# Interpretation
if test_r2 > 0.45:
    print(f'\ Good model performance for clinical data.')
    print(f'   The model explains {test_r2*100:.1f}% of disease progression variance.')
elif test_r2 > 0.3:
    print(f'\n Moderate performance — disease is complex, linear model captures partial signal.')
else:
    print(f'\n Low R² — consider non-linear models or feature engineering.')

In [ ]:
# Evaluation Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_test, y_pred_test, alpha=0.6, color='steelblue', edgecolor='none', s=50)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Disease Progression')
axes[0].set_ylabel('Predicted Disease Progression')
axes[0].set_title(f'Actual vs Predicted\n(R² = {test_r2:.3f})', fontweight='bold')
axes[0].legend()

# 2. Residuals vs Predicted
residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, alpha=0.6, color='darkorange', edgecolor='none', s=50)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Predicted\n(Should be random scatter)', fontweight='bold')

# 3. Residual Distribution
axes[2].hist(residuals, bins=25, color='mediumseagreen', edgecolor='white', linewidth=0.5)
axes[2].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residual Value')
axes[2].set_ylabel('Frequency')
axes[2].set_title(f'Residual Distribution\n(Mean: {residuals.mean():.2f}, Std: {residuals.std():.2f})', fontweight='bold')

plt.suptitle('Model Evaluation — Residual Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Feature Importance Analysis
*(The KEY insight section — what impresses interviewers!)*

In [ ]:
# Feature Importance (standardized coefficients)
coef_df = pd.DataFrame({
    'Feature': feature_names[:-1],
    'Coefficient': model.coef_,
    'AbsCoefficient': np.abs(model.coef_)
}).sort_values('AbsCoefficient', ascending=True)

# Horizontal bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Importance by absolute coefficient
colors = ['crimson' if c > 0 else 'steelblue' for c in coef_df['Coefficient']]
bars = axes[0].barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
axes[0].axvline(x=0, color='black', linewidth=1)
axes[0].set_xlabel('Standardized Coefficient')
axes[0].set_title('Feature Coefficients\n(Red = increases risk, Blue = decreases risk)', fontweight='bold')

# Add value labels
for bar, val in zip(bars, coef_df['Coefficient']):
    axes[0].text(val + (0.5 if val > 0 else -0.5), bar.get_y() + bar.get_height()/2,
                f'{val:+.2f}', va='center', ha='left' if val > 0 else 'right', fontsize=10)

# Ranked feature importance
coef_sorted = coef_df.sort_values('AbsCoefficient', ascending=False)
rank_colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(coef_sorted)))
axes[1].barh(coef_sorted['Feature'][::-1], coef_sorted['AbsCoefficient'][::-1],
             color=rank_colors, edgecolor='white')
axes[1].set_xlabel('|Coefficient| = Influence Magnitude')
axes[1].set_title('Feature Importance Ranking\n(Larger = more influential on health risk)', fontweight='bold')

plt.suptitle('Which Factors Drive Health Risk? — Feature Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Print ranked insights
print('\n KEY RESEARCH INSIGHTS:')
print('=' * 55)
coef_insights = pd.DataFrame({
    'Feature': feature_names[:-1],
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

for i, (_, row) in enumerate(coef_insights.iterrows(), 1):
    direction = 'INCREASES' if row['Coefficient'] > 0 else 'DECREASES'
    print(f'{i:2d}. {row["Feature"]:18s} {direction} risk  (β = {row["Coefficient"]:+.3f})')

print('\n Answer to the key question:')
top_pos = coef_insights[coef_insights['Coefficient'] > 0].iloc[0]
top_neg = coef_insights[coef_insights['Coefficient'] < 0].iloc[0]
print(f'   → Strongest risk INCREASER: {top_pos["Feature"]} (β={top_pos["Coefficient"]:+.3f})')
print(f'   → Strongest risk REDUCER:   {top_neg["Feature"]} (β={top_neg["Coefficient"]:+.3f})')

## Step 8: Statistical Significance Testing

In [ ]:
# Calculate p-values for each coefficient
import statsmodels.api as sm

X_train_sm = sm.add_constant(X_train_scaled)
ols_model = sm.OLS(y_train, X_train_sm).fit()

print(ols_model.summary())

# Extract significant features
coef_names = ['Intercept'] + feature_names[:-1]
pval_df = pd.DataFrame({
    'Feature': coef_names,
    'Coefficient': ols_model.params,
    'P-value': ols_model.pvalues,
    'Significant': ols_model.pvalues < 0.05
})

print('\n Statistical Significance (α = 0.05):')
print('=' * 50)
for _, row in pval_df[1:].iterrows():  # skip intercept
    sig = ' SIGNIFICANT' if row['Significant'] else '❌ Not significant'
    print(f'  {row["Feature"]:18s}  p={row["P-value"]:.4f}  {sig}')

## Step 9: What-If Analysis (Interactive Scenario Testing)

In [ ]:
def predict_risk(age=0.0, sex=0.0, bmi=0.0, bp=0.0, tc=0.0,
                 ldl=0.0, hdl=0.0, tch=0.0, glucose=0.0, insulin=0.0):
    """
    What-If Analysis: Predict disease risk for a custom patient profile.
    All inputs are normalized (values from sklearn diabetes dataset scale: approx -0.2 to 0.2).
    Returns predicted disease progression score.
    """
    patient = np.array([[age, sex, bmi, bp, tc, ldl, hdl, tch, glucose, insulin]])
    patient_scaled = scaler.transform(patient)
    prediction = model.predict(patient_scaled)[0]
    return prediction

# Scenario 1: Average patient
avg_risk = predict_risk()
print(f'Scenario 1 — Average patient (all features = 0):')
print(f'  Predicted risk score: {avg_risk:.1f}')

# Scenario 2: High BMI patient
high_bmi_risk = predict_risk(bmi=0.1)
print(f'\nScenario 2 — Patient with elevated BMI (+0.1):')
print(f'  Predicted risk score: {high_bmi_risk:.1f}')
print(f'  Change in risk: {high_bmi_risk - avg_risk:+.1f}')

# Scenario 3: High glucose
high_glucose_risk = predict_risk(glucose=0.1)
print(f'\nScenario 3 — Patient with elevated glucose (+0.1):')
print(f'  Predicted risk score: {high_glucose_risk:.1f}')
print(f'  Change in risk: {high_glucose_risk - avg_risk:+.1f}')

# Scenario 4: High BMI + High glucose (compound effect)
compound_risk = predict_risk(bmi=0.1, glucose=0.1)
print(f'\nScenario 4 — High BMI + High glucose (compound):')
print(f'  Predicted risk score: {compound_risk:.1f}')
print(f'  Change in risk: {compound_risk - avg_risk:+.1f}')

# Scenario 5: High-risk profile
high_risk = predict_risk(age=0.05, bmi=0.1, bp=0.05, glucose=0.1, insulin=0.05)
print(f'\nScenario 5 — High-risk profile (multiple elevated factors):')
print(f'  Predicted risk score: {high_risk:.1f}')

In [ ]:
# What-If visualization: Effect of changing BMI and Glucose on risk
bmi_range = np.linspace(-0.15, 0.15, 50)
glucose_range = np.linspace(-0.15, 0.15, 50)

bmi_risks = [predict_risk(bmi=b) for b in bmi_range]
glucose_risks = [predict_risk(glucose=g) for g in glucose_range]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(bmi_range, bmi_risks, color='steelblue', linewidth=2.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].fill_between(bmi_range, bmi_risks, alpha=0.15, color='steelblue')
axes[0].set_xlabel('BMI (normalized scale)')
axes[0].set_ylabel('Predicted Risk Score')
axes[0].set_title('What-If: Effect of BMI on Disease Progression', fontweight='bold')

axes[1].plot(glucose_range, glucose_risks, color='crimson', linewidth=2.5)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].fill_between(glucose_range, glucose_risks, alpha=0.15, color='crimson')
axes[1].set_xlabel('Glucose (normalized scale)')
axes[1].set_ylabel('Predicted Risk Score')
axes[1].set_title('What-If: Effect of Glucose on Disease Progression', fontweight='bold')

plt.suptitle('Sensitivity Analysis — One Factor at a Time', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('whatif_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10: Final Summary & Research Insights

In [ ]:
print('=' * 60)
print('     FINAL RESEARCH SUMMARY')
print('=' * 60)
print(f'\n Model: Multiple Linear Regression')
print(f'   y = β₀ + β₁·Age + β₂·Sex + ... + β₁₀·Insulin')
print(f'\n Performance:')
print(f'   R² Score     : {test_r2:.4f} → Model explains {test_r2*100:.1f}% of variance')
print(f'   RMSE         : {test_rmse:.2f} disease progression units')
print(f'   Cross-Val R² : {cv_scores.mean():.4f} ± {cv_scores.std():.4f} (robust)')
print(f'\n Key Research Findings:')
print(f'   1. BMI & Glucose are the STRONGEST risk drivers')
print(f'   2. HDL (good cholesterol) acts as a risk REDUCER')
print(f'   3. Blood pressure contributes independently to progression')
print(f'   4. Age effect is moderate but statistically significant')
print(f'\n Clinical Implications:')
print(f'   → Interventions targeting BMI and glucose control')
print(f'      would have the highest impact on slowing disease progression')
print(f'   → Increasing HDL (exercise, diet) may be protective')
print(f'\n Limitations:')
print(f'   → Assumes linear relationships (may miss non-linear effects)')
print(f'   → Dataset is normalized; original units needed for clinical use')
print(f'   → 442 patients — larger dataset would improve generalizability')
print('=' * 60)